# Stage 2 — screen

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and an inline eval. The implementation itself is yours to write.


## 1. Setup


Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running anything else in the same runtime), run them top-to-bottom.


### Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### Install dependencies


Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### Bridge your OpenAI key


Add `OPENAI_API_KEY` in Colab's Secrets panel (key icon, left sidebar) and toggle notebook access first.


In [ ]:
# OpenAI key bridge: Colab's userdata.get() does NOT populate os.environ,
# but our scripts read os.environ["OPENAI_API_KEY"]. Bridge it once.
# Add the key in Colab via the left sidebar → "Secrets" (key icon) → name it OPENAI_API_KEY.
import os
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY set in os.environ")
    else:
        print("WARNING: OPENAI_API_KEY secret is empty — Stage 2/5 and *_llm.py evals will fail")
except Exception as e:
    print("Not running in Colab or userdata unavailable; set OPENAI_API_KEY yourself.")
    print("Detail:", e)


### Seed prior stages' outputs from the reference run


Stage 2 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_01/data/` from the canonical reference run so Stage 2 has inputs to work with — but only when the dir is empty, so re-running an earlier stage IN THIS runtime is not clobbered.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 2.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <2 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 2):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Configure inputs


The CRITERIA string is the prompt the LLM sees. Edit it to change what gets included. (Also persisted to `stage_02/input.txt`/`criteria.txt` so the standalone `eval/eval_02_llm.py` keeps working.)


In [ ]:
MODEL = "gpt-5.4-nano"
SLEEP_SECONDS = 1.0

CRITERIA = '''\
Include: primary research papers (2025-2026) reporting at least one
in-vivo mammalian study arm in support of monoclonal antibody (mAb)
development. Eligible study arms include:
  - pharmacokinetics (PK) or toxicokinetics
  - single-dose or repeat-dose toxicology
  - immunogenicity / anti-drug antibody (ADA) assessment
  - tissue biodistribution
  - tissue cross-reactivity confirmed in vivo

Eligible species: mouse, rat, cynomolgus monkey, rhesus monkey, dog,
rabbit, minipig.

Exclude:
  - reviews, meta-analyses, perspectives, commentaries, editorials
  - papers reporting only in-vitro binding, cell-line, or PBMC work
    with no in-vivo arm
  - veterinary mAb studies (animal as patient, not as preclinical model)
  - discovery-stage efficacy-only papers using mouse xenograft tumor
    models with no PK/tox/immunogenicity arm
  - mAb-conjugate papers where the conjugate is the primary subject
  - case reports of mAb adverse events in patients
'''

# Persist for backward compat with the standalone eval_02_llm.py.
import os
os.makedirs("stage_02", exist_ok=True)
with open("stage_02/criteria.txt", "w") as f: f.write(CRITERIA)
with open("stage_02/input.txt", "w") as f:
    f.write(f"MODEL = {MODEL}\nCRITERIA_FILE = stage_02/criteria.txt\n"
            f"SLEEP_SECONDS = {SLEEP_SECONDS}\n")
print(f"MODEL={MODEL}  SLEEP={SLEEP_SECONDS}s  criteria: {len(CRITERIA)} chars")


## 3. Spec — paste this into Gemini


Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Write Python that:

1. Reads `stage_01/data/pmids.json` and pulls out the `records` list.
2. For each record, calls OpenAI `chat.completions.create` with the
   MODEL from the inputs cell, the CRITERIA string inlined into the
   prompt, and the record's title + abstract. Use
   `temperature=0` and `response_format={"type": "json_object"}`.
3. Parses each response as JSON with shape
   `{"verdict": "include"|"exclude", "rationale": "<one sentence>"}`.
4. Builds a list `screened` — one dict per record carrying
   `pmid`, `title`, `abstract`, `pub_types`, `verdict`, `rationale`.

Do NOT save to disk — the next cell handles persistence.
```


## 4. Gotchas Gemini probably won't know


Copy any that apply into Gemini if it goes off-track:

- **Use the OpenAI JSON-mode response format.** Pass
  `response_format={"type": "json_object"}` AND `temperature=0` so the
  output is parseable without regex.
- **`OpenAI()` reads the env var.** As long as you've bridged the
  Colab secret in Section 1, no explicit api_key arg.
- **Rate-limit yourself.** `time.sleep(SLEEP_SECONDS)` between
  requests (the free tier rate-limits aggressively).
- **Strip everything you don't need from the record before saving.**
  Carrying every metadata field forward bloats the output.


## 5. Seed — a few lines to anchor Gemini


In [ ]:
import json, os, time
from openai import OpenAI
client = OpenAI()  # reads OPENAI_API_KEY from os.environ

# Load Stage 1 records.
with open("stage_01/data/pmids.json") as f:
    records = json.load(f)["records"]

# Produce a variable named `screened` (list of dicts) for the inspect cell.


## 6. Your implementation


Drive Gemini to fill this in. Iterate until the inspect cell below shows reasonable output and the eval cell passes.


In [ ]:
# TODO: implement Stage 2 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the inspect + eval cells next.


## 7. Inspect output


In [ ]:
import json, os
os.makedirs("stage_02/data", exist_ok=True)
with open("stage_02/data/screened.json", "w") as f:
    json.dump(screened, f, indent=2)
n_inc = sum(1 for r in screened if r['verdict']=='include')
print(f"Saved {len(screened)} rows → stage_02/data/screened.json")
print(f"Verdict: {n_inc} include / {len(screened)-n_inc} exclude\n")
print("Sample (first 3 include + first 3 exclude):")
inc = [r for r in screened if r['verdict']=='include'][:3]
exc = [r for r in screened if r['verdict']=='exclude'][:3]
for r in inc + exc:
    tag = '✓' if r['verdict']=='include' else '✗'
    print(f"  {tag} PMID {r['pmid']}: {r['rationale'][:100]}")


## 8. Run eval


Inline eval — same checks as `eval/eval_02_script.py`, but the code is right here so you can see what it's measuring. Writes `stage_02/eval/eval_script.json` + `score.json`.


In [ ]:
# Same checks as eval/eval_02_script.py, inlined.
import json, os
os.makedirs("stage_02/eval", exist_ok=True)
with open("stage_02/data/screened.json") as f:
    recs = json.load(f)
checks = {
    "every_record_has_pmid":   all(r.get("pmid") for r in recs),
    "verdicts_valid":          all(r.get("verdict") in {"include","exclude"} for r in recs),
    "rationales_non_empty":    all((r.get("rationale") or "").strip() for r in recs),
}
print("Script checks:")
for k, v in checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")
with open("stage_02/eval/eval_script.json", "w") as f:
    json.dump({"script": checks}, f, indent=2)
n_pass = sum(1 for v in checks.values() if v); n_total = len(checks)
score_path = "stage_02/eval/score.json"
score = json.load(open(score_path)) if os.path.exists(score_path) else {}
score["script"] = {"passed": n_pass, "total": n_total,
                   "percent": round(100*n_pass/n_total, 1)}
with open(score_path, "w") as f: json.dump(score, f, indent=2)
print(f"\nScore: {n_pass}/{n_total} ({score['script']['percent']}%)")
print("\nOptional AI-grader (costs ~$0.02):")
print("  !python eval/eval_02_llm.py")


## 9. Stuck? Skip this stage


Copy the reference run's Stage 2 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil
os.makedirs("stage_02/data", exist_ok=True)
shutil.copy("reference_outputs/stage_02/data/screened.json",
            "stage_02/data/screened.json")
print("copied reference Stage 2 output")
